<a href="https://colab.research.google.com/github/betulbilhan2/ai-carbon-tracker/blob/main/TerkenTech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Güncel ve Colab'ın Python sürümüne uyumlu PyTorch ve TabNet kurulumu
!pip install -q torch torchvision torchaudio
!pip install -q pandas numpy pyarrow scikit-learn pytorch-tabnet

import pandas as pd
import numpy as np
import os
import json
from sklearn.model_selection import train_test_split

print("Kütüphaneler başarıyla yüklendi!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.0 MB/s eta 0:00:00
Kütüphaneler başarıyla yüklendi!


In [ ]:
# 'verisetleri/' klasöründen tek ana karbon emisyonu dosyamızı yüklüyoruz
df_carbon_master = pd.read_csv('verisetleri/temizlenmis_carbon_emission (1).csv')

# Tam kopyaları temizleme güvencesi
df_carbon_master = df_carbon_master.drop_duplicates()

print(f"-> Ana Master Carbon Emission veri seti başarıyla yüklendi!")
print(f"Toplam Satır ve Sütun Sayısı: {df_carbon_master.shape}")
display(df_carbon_master.head(2))

-> Ana Master Carbon Emission veri seti başarıyla yüklendi!
Toplam Satır ve Sütun Sayısı: (10000, 20)


,Body Type,Sex,Diet,How Often Shower,Heating Energy Source,Transport,Vehicle Type,Social Activity,Monthly Grocery Bill,Frequency of Traveling by Air,Vehicle Monthly Distance Km,Waste Bag Size,Waste Bag Weekly Count,How Long TV PC Daily Hour,How Many New Clothes Monthly,How Long Internet Daily Hour,Energy efficiency,Recycling,Cooking_With,CarbonEmission
0,2,0,1,0,0,1,0,1,230,0,210.0,1,4,7,26,1,0,2,13,2238
1,1,0,3,1,2,2,0,1,114,2,9.0,0,3,9,38,5,0,2,9,1892


In [ ]:
# 1. Carbon Emission Yıllık Varsayım Kontrolü
min_val = df_carbon_master['CarbonEmission'].min()
max_val = df_carbon_master['CarbonEmission'].max()
mean_val = df_carbon_master['CarbonEmission'].mean()

print(f"CarbonEmission İstatistikleri -> Min: {min_val}, Max: {max_val}, Ortalama: {mean_val:.2f}")

# 2. Recycling ve Cooking_With Bitmask Kontrolü
if 'Recycling' in df_carbon_master.columns:
    print("\n--- Recycling Sütunu Dağılımı ---")
    print(df_carbon_master['Recycling'].value_counts().sort_index())

if 'Cooking_With' in df_carbon_master.columns:
    print("\n--- Cooking_With Sütunu Dağılımı ---")
    print(df_carbon_master['Cooking_With'].value_counts().sort_index())

CarbonEmission İstatistikleri -> Min: 306, Max: 8377, Ortalama: 2269.15

--- Recycling Sütunu Dağılımı ---
Recycling
0     645
1     587
2     625
3     647
4     616
5     589
6     637
7     588
8     648
9     633
10    619
11    626
12    633
13    630
14    602
15    675
Name: count, dtype: int64

--- Cooking_With Sütunu Dağılımı ---
Cooking_With
0     593
1     623
2     621
3     625
4     638
5     649
6     607
7     628
8     652
9     625
10    596
11    637
12    628
13    670
14    605
15    603
Name: count, dtype: int64


In [ ]:
import os
import json
from sklearn.model_selection import train_test_split

# 1. Çıktı klasörlerini oluşturalım
os.makedirs('01_processed/lookup_tables', exist_ok=True)

# 2. Lookup (Kütüphane) Tablolarını 'verisetleri/' klasöründen okuyup Parquet Olarak Kaydetme
df_food = pd.read_csv('verisetleri/temizlenmis_Food_Production.csv')
df_food[['Food product', 'Total_emissions']].to_parquet('01_processed/lookup_tables/food_emission_lookup.parquet', index=False)

df_fuel = pd.read_csv('verisetleri/temizlenmis_MY2022 Fuel Consumption Ratings.csv')
df_fuel[['Model', 'Engine Size(L)', 'Fuel Consumption (City (L/100 km)']].to_parquet('01_processed/lookup_tables/fuel_emission_lookup.parquet', index=False)

df_countries = pd.read_csv('verisetleri/temizlenmis_countries.csv')
df_countries.to_parquet('01_processed/lookup_tables/country_baseline.parquet', index=False)

# 3. Model Eğitimi İçin Train (%70) / Val (%15) / Test (%15) Setlerine Bölme
train_val, test = train_test_split(df_carbon_master, test_size=0.15, random_state=42)
train, val = train_test_split(train_val, test_size=0.1764, random_state=42)

train.to_parquet('01_processed/tabnet_train.parquet', index=False)
val.to_parquet('01_processed/tabnet_val.parquet', index=False)
test.to_parquet('01_processed/tabnet_test.parquet', index=False)

# 4. feature_metadata.json Dosyasını Oluşturma
metadata = {
    "carbon_emission_period": "annual (varsayım, doğrulandı - ortalama ~2200-4000 kg seviyesinde)",
    "recycling_encoding": "Bitmask (0-15 arası 4 bileşenli toplam - doğrulandı)",
    "cooking_with_encoding": "Bitmask (0-15 arası enerji yoğun yöntemler - doğrulandı)",
    "features": list(df_carbon_master.columns)
}

with open("01_processed/feature_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=4)

print("Tüm lookup dosyaları, train/val/test setleri ve feature_metadata.json başarıyla oluşturuldu!")
print("TEBRİKLER! Notebook 1 resmi olarak tamamlandı. 🚀")

Tüm lookup dosyaları, train/val/test setleri ve feature_metadata.json başarıyla oluşturuldu!
TEBRİKLER! Notebook 1 resmi olarak tamamlandı. 🚀
